# 📊 Análise de Vendas por Região e Produto

**Objetivo:** Entender o desempenho comercial da empresa identificando quais produtos e regiões geram mais lucro, onde estão as melhores oportunidades e como o ROI se distribui geograficamente.

**Dataset:** Base de dados interna de vendas (PI)  
**Ferramenta:** Python (pandas, matplotlib, seaborn)  
**Autor:** Darlan

---

## Perguntas de negócio que vamos responder:
1. Quais meses concentram mais vendas?
2. Quais produtos são mais vendidos?
3. Quais regiões têm melhor desempenho em lucro e ROI?
4. Qual produto performa melhor dentro de cada região?
5. Há produtos com lucro médio muito abaixo do esperado?

## 1. Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo visual padronizado
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

# Carregamento
vendas_df = pd.read_excel('data base/base dados PI.xlsx')

print(f'Shape do dataset: {vendas_df.shape}')
vendas_df.head()

## 2. Visão Geral do Dataset

In [ ]:
print('=== Tipos de dados ===')
print(vendas_df.dtypes)
print('\n=== Valores nulos ===')
print(vendas_df.isnull().sum())
print('\n=== Estatísticas descritivas ===')
vendas_df[['Lucro (R$)', 'ROI %']].describe().round(2)

## 3. Distribuição de Vendas por Mês

> Entender a sazonalidade ajuda no planejamento de estoque e campanhas de marketing.

In [ ]:
mes_venda = vendas_df['Mês'].value_counts().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(mes_venda.index, mes_venda.values, color=sns.color_palette('muted', len(mes_venda)))
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Volume de Vendas por Mês', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Número de Vendas')
plt.tight_layout()
plt.show()

print(f'\nMês com mais vendas: {mes_venda.idxmax()} ({mes_venda.max()} vendas)')
print(f'Mês com menos vendas: {mes_venda.idxmin()} ({mes_venda.min()} vendas)')

## 4. Análise por Produto

> Identificamos quais produtos dominam em volume e quais entregam melhor lucro.

In [ ]:
produto_venda = vendas_df['Produto'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume de vendas
bars = axes[0].barh(produto_venda.index, produto_venda.values,
                    color=sns.color_palette('muted', len(produto_venda)))
axes[0].bar_label(bars, padding=3, fontsize=9)
axes[0].set_title('Volume de Vendas por Produto', fontweight='bold')
axes[0].set_xlabel('Número de Vendas')
axes[0].invert_yaxis()

# Lucro médio por produto
lucro_medio = vendas_df.groupby('Produto')['Lucro (R$)'].mean().sort_values(ascending=True)
bars2 = axes[1].barh(lucro_medio.index, lucro_medio.values,
                     color=sns.color_palette('coolwarm', len(lucro_medio)))
axes[1].bar_label(bars2, fmt='R$ %.0f', padding=3, fontsize=9)
axes[1].set_title('Lucro Médio por Produto (R$)', fontweight='bold')
axes[1].set_xlabel('Lucro Médio (R$)')

plt.tight_layout()
plt.show()

In [ ]:
analise_lucro = vendas_df.groupby('Produto')['Lucro (R$)'].agg(
    Média='mean',
    Máximo='max',
    Mínimo='min',
    Total='sum',
    Vendas='count'
).round(2).sort_values('Total', ascending=False)

print('=== Análise de Lucro por Produto ===')
analise_lucro

## 5. Desempenho por Micro Região

> Comparar regiões permite alocar melhor recursos de vendas e logística.

In [ ]:
comparativo_regiao = vendas_df.groupby('Micro Região').agg(
    Total_Vendas=('Produto', 'count'),
    Lucro_Total=('Lucro (R$)', 'sum'),
    Lucro_Medio=('Lucro (R$)', 'mean'),
    ROI_Medio=('ROI %', 'mean')
).round(2).sort_values('Lucro_Total', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lucro total por região
regioes = comparativo_regiao.index
cores = sns.color_palette('muted', len(regioes))

bars = axes[0].bar(regioes, comparativo_regiao['Lucro_Total'], color=cores)
axes[0].bar_label(bars, fmt='R$ %.0f', padding=3, fontsize=8, rotation=0)
axes[0].set_title('Lucro Total por Região (R$)', fontweight='bold')
axes[0].set_xlabel('Micro Região')
axes[0].set_ylabel('Lucro (R$)')
axes[0].tick_params(axis='x', rotation=30)

# ROI médio por região
bars2 = axes[1].bar(regioes, comparativo_regiao['ROI_Medio'], color=cores)
axes[1].bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=8)
axes[1].set_title('ROI Médio por Região (%)', fontweight='bold')
axes[1].set_xlabel('Micro Região')
axes[1].set_ylabel('ROI Médio (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('\n=== Tabela Comparativa por Região ===')
comparativo_regiao

## 6. Produto x Região — Onde cada produto performa melhor?

> Este cruzamento revela oportunidades regionais específicas por produto.

In [ ]:
pivot_lucro = vendas_df.pivot_table(
    values='Lucro (R$)',
    index='Micro Região',
    columns='Produto',
    aggfunc='sum'
).fillna(0).round(0)

plt.figure(figsize=(12, 6))
sns.heatmap(
    pivot_lucro,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Lucro Total (R$)'}
)
plt.title('Lucro Total por Produto e Região (R$)', fontsize=14, fontweight='bold')
plt.xlabel('Produto')
plt.ylabel('Micro Região')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Produto campeão por região
produtos_por_regiao = vendas_df.groupby(['Micro Região', 'Produto']).agg(
    Total_Vendas=('Produto', 'count'),
    Lucro_Total=('Lucro (R$)', 'sum')
).reset_index()

top_por_regiao = (
    produtos_por_regiao
    .sort_values('Lucro_Total', ascending=False)
    .groupby('Micro Região')
    .first()
    .reset_index()
    .rename(columns={'Produto': 'Melhor Produto', 'Lucro_Total': 'Lucro (R$)'})
)

print('=== Produto com maior lucro em cada região ===')
top_por_regiao[['Micro Região', 'Melhor Produto', 'Total_Vendas', 'Lucro (R$)']]

## 7. Conclusões e Recomendações

Com base na análise realizada, destacamos os principais insights:

### 📌 Sazonalidade
- Identificamos os meses de pico e baixa — úteis para planejamento de campanhas e estoque.

### 📌 Produtos
- Nem sempre o produto mais vendido é o mais lucrativo. Focar apenas em volume pode esconder oportunidades de margem.
- Produtos com lucro médio baixo merecem revisão de precificação ou custo.

### 📌 Regiões
- Existe variação significativa de ROI entre as micro regiões — vale investigar os fatores que explicam essa diferença (logística, perfil de cliente, mix de produtos).
- A região com maior lucro total nem sempre é a de maior ROI, o que indica que escala e eficiência são dinâmicas diferentes.

### 📌 Próximos passos sugeridos
1. Cruzar dados com custo logístico por região para ajustar o ROI real
2. Analisar tendência mensal de lucro (não só volume)
3. Aplicar segmentação de clientes por região para personalizar ofertas